# آموزش 4DGS و ارزیابی TIDES روی کل سکانس EDS

این نوت‌بوک جدا برای کل `00_peanuts_dark` است، نه کلیپ ۱٫۵ثانیه‌ای.
فایل را در پوشه `notebooks` مخزن `TIDES_2026` روی سرور بگذارید، کرنل **Python (tides)**
را انتخاب کنید و **Run All** بزنید. سلول اول کد شامل مسیرهای شماست؛ نیازی به افزودن سلول نیست.

**Run All آماده‌سازی و سپس آموزش واقعی GPU را شروع می‌کند**؛ پس از آن شبیه‌سازی و ارزیابی
دو حالت TIDES اجرا می‌شوند. baselineها و ablationها در این اجرا انتخاب نشده‌اند.
خروجی جداگانه در `work/eds/00_peanuts_dark_full` ذخیره می‌شود.
اگر صحنه کامل از قبل آماده باشد دوباره ساخته نمی‌شود؛ چک‌پوینت کامل نیز دوباره آموزش نمی‌بیند.
خروجی ناقص خودکار حذف یا بازنویسی نمی‌شود.

همه فریم‌های داخل بازه پوز معتبر، با stride=1 انتخاب می‌شوند؛ محدودیت ۱٫۵ ثانیه و سقف ۶۰۰
فریم وجود ندارد. تفکیک قبلی آموزش/ارزیابی و تنظیمات ۳۰۰۰ coarse + ۱۴۰۰۰ fine حفظ شده‌اند.
نتیجه هنوز مربوط به یک سکانس است و معادل میانگین کل EDS مقاله نیست.
قرارداد GT=RGB و وضعیت flip همان فرض‌های ثبت‌شده قبلی هستند؛ تغییر خودکاری در آن‌ها نمی‌دهیم.


In [ ]:
# تنظیمات از قبل برای سرور شما؛ اجرای سلول‌های اصلی نیاز به export ترمینال ندارد.
import os
os.environ['TIDES_EDS_ROOT'] = '/u/marmary/evtween/EDS/EDS'
os.environ['TIDES_EDS_CALIB'] = '/u/marmary/evtween/EDS/calib/00_calib/camchain-mediajaviJAVISdatasetshwdscalibratione2kalibr.yaml'
os.environ['TIDES_SEQUENCE'] = '00_peanuts_dark'
os.environ['TIDES_WORK'] = '/u/marmary/airlab_v2e/TIDES_2026/work/eds/00_peanuts_dark_full'
os.environ['TIDES_FOURDGS'] = '/u/marmary/airlab_v2e/TIDES/third_party/4DGaussians'
os.environ['TIDES_GT_POSE_FRAME'] = 'rgb'
os.environ['TIDES_TANGENT_MODE'] = 'reverse'
os.environ['TIDES_SEED'] = '0'
os.environ['TIDES_RUN_STAGES'] = 'prepare,audit,train,tides,evaluate,record'
# A previous clip session must not redirect full-sequence metrics or bounds.
os.environ.pop('TIDES_T_START', None)
os.environ.pop('TIDES_T_END', None)
os.environ.pop('TIDES_METRICS', None)
print('Full-sequence workflow selected. Training/simulation start when their cells run.')


In [ ]:
from pathlib import Path
import hashlib, importlib.metadata, json, os, runpy, shlex, subprocess, sys
from IPython.display import display
ROOT = next((p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
             if (p/'scripts'/'prepare_scene.py').is_file() and (p/'tides').is_dir()), None)
if ROOT is None:
    raise RuntimeError('Open Jupyter inside the TIDES_2026 checkout or its notebooks folder')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
def run(*cmd):
    cmd = [str(v) for v in cmd]
    print('$', shlex.join(cmd), flush=True)
    with subprocess.Popen(cmd, cwd=ROOT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          text=True, bufsize=1) as process:
        for line in process.stdout:
            print(line, end='', flush=True)
        code = process.wait()
    if code:
        raise subprocess.CalledProcessError(code, cmd)
def sha(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()
def git(*args, path=ROOT):
    r = subprocess.run(['git', '-C', str(path), *args], capture_output=True, text=True)
    return r.stdout.strip() if r.returncode == 0 else None
print('Repository:', ROOT, '\nKernel:', sys.executable)


## 1. مسیرهای آماده و پیش‌نیازها

مسیر checkout قبلی 4DGaussians و حالت reverse از قبل تنظیم شده‌اند.
هیچ وابستگی‌ای خودکار نصب نمی‌شود. صحنه کوتاه قبلی در پوشه دیگری باقی می‌ماند.
این نوت‌بوک به همان اسکریپت‌های اصلاح‌شده مخزن نیاز دارد (کامیت 8f612d9 یا نسخه بعد از آن).


In [ ]:
EDS_ROOT = Path(os.environ.get('TIDES_EDS_ROOT', '/u/marmary/evtween/EDS/EDS')).expanduser().resolve()
EDS_CALIB = Path(os.environ.get('TIDES_EDS_CALIB', '/u/marmary/evtween/EDS/calib/00_calib/camchain-mediajaviJAVISdatasetshwdscalibratione2kalibr.yaml')).expanduser().resolve()
SEQUENCE = os.environ.get('TIDES_SEQUENCE', '00_peanuts_dark')
WORK = Path(os.environ.get('TIDES_WORK', str(ROOT/'work'/'eds'/SEQUENCE))).expanduser().resolve()
FOURDGS = Path(os.environ.get('TIDES_FOURDGS', str(ROOT/'third_party'/'4DGaussians'))).expanduser().resolve()
CFG4D = ROOT/'configs'/'4dgs'/'tides_default.py'
THIRD_PARTY = Path(os.environ.get('TIDES_THIRD_PARTY', str(ROOT/'third_party'))).expanduser().resolve()
T_START, T_END = 0.0, None  # actual supported endpoints are read after preparation
SEED = int(os.environ.get('TIDES_SEED', '0'))  # preparation + TIDES; upstream training has its own seeding
GT_POSE_FRAME = os.environ.get('TIDES_GT_POSE_FRAME', 'rgb')
TANGENT_MODE = os.environ.get('TIDES_TANGENT_MODE', 'reverse')
if TANGENT_MODE not in {'forward','reverse'}:
    raise ValueError('TIDES_TANGENT_MODE must be forward or reverse')
STAGES = {s.strip() for s in os.environ.get('TIDES_RUN_STAGES', '').split(',') if s.strip()}
ALLOWED = {'inspect','prepare','audit','train','tides','baselines','evaluate','ablations','tables','record'}
if STAGES - ALLOWED:
    raise ValueError(f'Unknown stages: {STAGES - ALLOWED}')
if GT_POSE_FRAME not in {'rgb','event'} or (T_END is not None and not T_START < T_END):
    raise ValueError('Invalid pose frame or time range')
os.environ['TIDES_EDS_CALIB'] = str(EDS_CALIB)  # also used by evaluation of legacy prepared scenes
METHODS = ['esim','v2e','iebcs','dvs_voltmeter','tides_10x','tides_adaptive']
print(json.dumps({'EDS':str(EDS_ROOT),'calibration':str(EDS_CALIB),'work':str(WORK),
                  '4DGS':str(FOURDGS),'tangent_mode':TANGENT_MODE,'stages':sorted(STAGES)}, indent=2))


## 2. آماده‌سازی کل سکانس و کنترل پوشش فریم‌ها

آماده‌سازی بدون t-end و با سقف sys.maxsize اجرا می‌شود. بعد از آماده‌سازی، بازه زمانی
از tides_meta.json خوانده می‌شود؛ همان کاری که سلول تنظیمات دستی قرار بود انجام دهد،
اما اینجا پس از ساخته‌شدن فایل اجرا می‌شود. تعداد و زمان فریم‌ها با کل بازه معتبر دیتاست
مقایسه می‌شود تا صحنه کوتاه یا آماده‌سازی ناقص به‌اشتباه به عنوان اجرای کامل استفاده نشود.


In [ ]:
# Check prerequisites before creating a new scene or starting training.
import torch
for required in [EDS_ROOT/SEQUENCE, EDS_CALIB, FOURDGS/'train.py',
                 FOURDGS/'scene'/'dataset_readers.py', CFG4D,
                 ROOT/'scripts'/'audit_prepared_scene.py', ROOT/'scripts'/'compare_eds_paper.py']:
    if not required.exists():
        raise FileNotFoundError(f'Missing required path: {required}')
if not torch.cuda.is_available():
    raise RuntimeError('Select the tides CUDA kernel before running the full workflow')
print('GPU:', torch.cuda.get_device_name(0))

def prepare_full_sequence():
    meta_path = WORK/'tides_meta.json'
    if meta_path.is_file():
        print('Reusing prepared scene; validating full coverage below:', WORK)
    elif 'prepare' in STAGES:
        if (WORK/'scene').exists():
            raise FileExistsError('Incomplete scene without metadata; use a fresh WORK, do not overwrite it')
        run(sys.executable, ROOT/'scripts'/'prepare_scene.py', '--dataset','eds','--root',EDS_ROOT,
            '--seq',SEQUENCE,'--out',WORK,'--calib',EDS_CALIB,'--gt-pose-frame',GT_POSE_FRAME,
            '--t-start',0,'--stride',1,'--max-frames',sys.maxsize,'--seed',SEED)
    else:
        raise FileNotFoundError('No prepared full scene. Enable prepare or select its WORK.')
    return json.loads(meta_path.read_text())

prepared = prepare_full_sequence()
T_START, T_END = map(float, prepared['t_range'])
os.environ['TIDES_T_START'], os.environ['TIDES_T_END'] = str(T_START), str(T_END)
print('Full-scene selected frames:', len(prepared['frame_times_s']))
print('Actual supported time range:', [T_START,T_END])

# Loading sequence metadata does not load the full event stream.
import numpy as np
from tides.datasets import load_sequence
original = load_sequence('eds', str(EDS_ROOT), SEQUENCE,
                         calib_path=str(EDS_CALIB), gt_pose_frame=GT_POSE_FRAME)
if original.gt_poses is None:
    raise ValueError('Original EDS GT poses are required')
times = original.frame_times
lo, hi = original.gt_poses[0][0], original.gt_poses[0][-1]
expected = times[(times >= 0) & (times >= lo) & (times <= hi)]
actual = np.asarray(prepared['frame_times_s'])
if len(expected) != len(actual) or not np.allclose(expected, actual, rtol=0, atol=1e-7):
    raise ValueError('Existing WORK does not cover every frame in the valid GT interval. Use a fresh full WORK.')
print('Source RGB frames:', len(times), '| selected:', len(actual), '| outside valid GT:', len(times)-len(actual))

def check_scene():
    run(sys.executable, ROOT/'scripts'/'audit_prepared_scene.py','--work',WORK,
        '--calib',EDS_CALIB,'--fourdgs',FOURDGS)
    meta = json.loads((WORK/'tides_meta.json').read_text())
    if meta['dataset'] != 'eds' or meta['sequence'] != SEQUENCE or Path(meta['root']).resolve() != EDS_ROOT:
        raise ValueError('Prepared scene is from another dataset/root/sequence; choose the correct WORK')
    assumed = meta.get('loader_kwargs',{}).get('gt_pose_frame','rgb')
    if assumed != GT_POSE_FRAME:
        raise ValueError('Existing scene uses another GT camera assumption; choose a new WORK and prepare')
    if meta['t_range'][0] < T_START-1e-6 or meta['t_range'][1] > T_END+1e-6:
        raise ValueError('Prepared scene lies outside the requested interval')
    print('Actual selected interval:', meta['t_range'], 'frames:', len(meta['frame_times_s']))
    return meta
if STAGES & {'audit','train','tides','baselines','evaluate','ablations'}:
    META = check_scene()
else:
    print('No stages selected. Prepared metadata exists:', (WORK/'tides_meta.json').is_file())


## 3. Preview or explicitly start training

The command uses the existing 4DGS pipeline and absolute paths. The config explicitly
sets white background, held-out every eighth view (`eval=True`), original resolution,
and disables progress rendering. These are repository choices, not paper-confirmed
hyperparameters. Upstream boolean CLI flags are switches; never append `False` as a value.
With no checkpoint, training starts from the prepared RGB scene (3000 coarse + 14000 fine iterations).
A complete checkpoint is reused. Incomplete existing output raises an error instead of overwriting it.
Live subprocess logs are displayed below; after training the selected simulation/evaluation stages continue.
Check the printed upstream revision and perform a GPU render/dependency check in your
`tides` environment before enabling this stage. No `--seed` flag is invented for upstream.


In [ ]:
def checkpoint():
    files = sorted((WORK/'4dgs'/'point_cloud').glob('iteration_*/point_cloud.ply'))
    files = [p for p in files if p.parent.name.removeprefix('iteration_').isdigit()
             and (p.parent/'deformation.pth').is_file()]
    if not files:
        raise FileNotFoundError('No fine-stage PLY + deformation checkpoint; a point_cloud directory alone is insufficient')
    return max(files, key=lambda p: int(p.parent.name.split('_')[-1])).parent

TRAIN_CMD = [sys.executable, FOURDGS/'train.py','-s',WORK/'scene','-m',WORK/'4dgs',
             '--configs',CFG4D,'--expname',f'eds_{SEQUENCE}','--port',os.environ.get('TIDES_PORT','6017')]
print('Training command (preview):', shlex.join([str(x) for x in TRAIN_CMD]))
if 'train' in STAGES:
    if (WORK/'4dgs').exists():
        print('Reusing existing complete checkpoint:', checkpoint())
    else:
        import torch
        if not torch.cuda.is_available():
            raise RuntimeError('4DGS training requires the configured CUDA environment')
        print('No checkpoint: fitting 4DGS from RGB, 3000 coarse + 14000 fine iterations.')
        run(*TRAIN_CMD)
        print('Training checkpoint:', checkpoint())


## 4. Simulation, baselines, evaluation, ablations (explicit opt-in)

All methods reuse the same trained scene and actual prepared interval.
The notebook defaults to exact reverse-mode time derivatives because upstream 4DGS
uses grid_sample, which may lack forward AD in the installed PyTorch. This is not
finite differencing and needs no retraining. Set TIDES_TANGENT_MODE=forward only in
a supported environment. The same mode is used for Chamfer speed estimation. Simulation can
also be expensive. Baseline adapters need their own upstream installs; ESIM can fall
back to the repository implementation (inspect its event JSON before claiming official
ESIM results). Metric assumptions include excess IG-NLL and diagonal-normalized Chamfer.
Legacy metric files without provenance are not silently merged: use a fresh output.


In [ ]:
COMMON = ['--work',WORK,'--fourdgs',FOURDGS,'--fourdgs-config',CFG4D]
TANGENT_ARGS = ['--tangent-mode',TANGENT_MODE]
if STAGES & {'tides','baselines','evaluate','ablations'}:
    print('Reusing checkpoint:', checkpoint())
if 'tides' in STAGES:
    for name in ['tides_10x','tides_adaptive']:
        run(sys.executable,ROOT/'scripts'/'simulate.py',*COMMON,*TANGENT_ARGS,'--config',ROOT/'configs'/(name+'.yaml'),
            '--name',name,'--seed',SEED)
if 'baselines' in STAGES:
    run(sys.executable,ROOT/'scripts'/'run_baselines.py',*COMMON,'--third-party',THIRD_PARTY,
        '--methods','esim','v2e','iebcs','dvs_voltmeter')
METRICS = Path(os.environ.get('TIDES_METRICS', str(WORK/'metrics.json'))).expanduser().resolve()
if 'evaluate' in STAGES:
    missing = [m for m in METHODS if not (WORK/'events'/(m+'.h5')).is_file()]
    if missing:
        print('Partial evaluation; missing methods:', missing)
    available = [m for m in METHODS if m not in missing]
    if not available:
        raise FileNotFoundError('No simulated event streams')
    run(sys.executable,ROOT/'scripts'/'evaluate.py',*COMMON,*TANGENT_ARGS,'--calib',EDS_CALIB,'--out',METRICS,'--methods',*available)
if 'ablations' in STAGES:
    run(sys.executable,ROOT/'scripts'/'run_ablations.py',*COMMON,*TANGENT_ARGS,'--seed',SEED)


## 5. Results, comparison and reproducibility record

Single-sequence results and available-run averages are diagnostic only. They cannot establish
paper reproduction or even matching rankings. Aggregation is opt-in and may contain
unequal sequence coverage per method; inspect the contributing runs and settings.
The tables show ours, the paper EDS mean, signed difference, percent difference and missing-run status
for Tables 1, 2 and 4. Selecting `record` also saves `comparison/paper_comparison.csv` and `.md`.
These numerical comparisons do not certify protocol equivalence.
The manifest records source revisions, dirty state, environment packages and config hashes;
`SEED` does not claim control of upstream training or every stochastic baseline.


In [ ]:
results = json.loads(METRICS.read_text()) if METRICS.is_file() else {}
if results:
    import pandas as pd
    display(pd.DataFrame.from_dict(results.get('methods',{}), orient='index'))
    if results.get('preparation_sha256') != sha(WORK/'tides_meta.json'):
        print('WARNING: Results have missing or mismatched preparation provenance; do not compare as a current run.')
    report_tools = runpy.run_path(str(ROOT/'scripts'/'compare_eds_paper.py'))
    rows = report_tools['comparison_rows'](results)
    comparison = pd.DataFrame(rows)
    scope = json.loads((WORK/'tides_meta.json').read_text())
    print('Sequence:', results.get('sequence'), 'actual interval:', scope['t_range'])
    print('PAPER REPRODUCTION: NOT ESTABLISHED. This full sequence is compared with EDS dataset means.')
    print('Difference = ours - paper. Missing/nonfinite results remain missing, never zero.')
    for table in ['Table 1','Table 2','Table 4']:
        print(table)
        display(comparison[comparison['table'] == table].drop(columns='table').reset_index(drop=True).fillna('—'))
    if 'record' in STAGES:
        report_tools['save_comparison'](results, WORK/'comparison')
        print('Saved comparison CSV and Markdown:', WORK/'comparison')
else:
    print('No results yet. No training or simulation is started by displaying this notebook.')
if 'tables' in STAGES:
    run(sys.executable,ROOT/'scripts'/'make_tables.py','--work-root',ROOT/'work','--out',ROOT/'results'/'tables.md')
if 'record' in STAGES:
    WORK.mkdir(parents=True, exist_ok=True)
    packages = {}
    for name in ['torch','numpy','scipy','h5py','PyYAML','opencv-python','pandas','mmcv','torchvision']:
        try: packages[name] = importlib.metadata.version(name)
        except importlib.metadata.PackageNotFoundError: packages[name] = None
    manifest = {'sequence':SEQUENCE,'root':str(EDS_ROOT),'calibration':str(EDS_CALIB),
        'calibration_sha256':sha(EDS_CALIB) if EDS_CALIB.is_file() else None,
        'work':str(WORK),'requested_range':[T_START,T_END],'coverage':'all RGB frames within valid GT range','gt_pose_frame':GT_POSE_FRAME,
        'tangent_mode':TANGENT_MODE,'preparation_and_tides_seed':SEED,'training_seed':'upstream-controlled; inspect recorded source revision',
        'tides_revision':git('rev-parse','HEAD'),'tides_dirty':git('status','--porcelain'),
        'fourdgs_revision':git('rev-parse','HEAD',path=FOURDGS),
        'fourdgs_dirty':git('status','--porcelain',path=FOURDGS),'python':sys.version,
        'packages':packages,'stages':sorted(STAGES),'training_command':[str(x) for x in TRAIN_CMD],
        'config_hashes':{str(p.relative_to(ROOT)):sha(p) for p in [CFG4D,ROOT/'configs'/'metrics.yaml',
             ROOT/'configs'/'tides_10x.yaml',ROOT/'configs'/'tides_adaptive.yaml',ROOT/'configs'/'baselines.yaml']},
        'metadata_sha256':sha(WORK/'tides_meta.json') if (WORK/'tides_meta.json').is_file() else None,
        'checkpoint':str(checkpoint()) if list((WORK/'4dgs'/'point_cloud').glob('iteration_*/point_cloud.ply')) else None,
        'caveat':'Independent implementation; exact paper results remain unvalidated.'}
    (WORK/'reproduction_manifest.json').write_text(json.dumps(manifest,indent=2)+'\n')
    print('Saved:',WORK/'reproduction_manifest.json')
